In [2]:
import pandas as pd
borough= pd.read_csv(r"C:\Jason\Ride share project\taxi_zone_lookup.csv")
print(borough.shape)
cols = ['hvfhs_license_num',
        'request_datetime', 
        'trip_miles',
        'trip_time', #in seconds
        'PULocationID', 
        'DOLocationID',
        'congestion_surcharge', 
        'shared_request_flag']
uber_df = pd.read_parquet(r"C:\Jason\Ride share project\fhvhv_tripdata_2025-01.parquet", columns=cols)
uber_df = uber_df[uber_df['hvfhs_license_num'] == 'HV0003'].copy()
print(uber_df.shape)
uber_df.head()


(265, 4)
(15356455, 8)


,hvfhs_license_num,request_datetime,trip_miles,trip_time,PULocationID,DOLocationID,congestion_surcharge,shared_request_flag
0,HV0003,2025-01-01 00:28:07,1.32,1259,148,211,2.75,N
2,HV0003,2025-01-01 00:28:22,13.43,2874,132,181,0.00,N
3,HV0003,2025-01-01 00:27:13,0.82,264,76,76,0.00,N
4,HV0003,2025-01-01 00:33:29,1.61,457,76,76,0.00,N
5,HV0003,2025-01-01 00:34:43,9.66,2073,112,89,0.00,N


In [3]:
print(borough.columns.tolist())
print(borough.head())


['LocationID', 'Borough', 'Zone', 'service_zone']
   LocationID        Borough                     Zone service_zone
0           1            EWR           Newark Airport          EWR
1           2         Queens              Jamaica Bay    Boro Zone
2           3          Bronx  Allerton/Pelham Gardens    Boro Zone
3           4      Manhattan            Alphabet City  Yellow Zone
4           5  Staten Island            Arden Heights    Boro Zone


In [4]:
#merge pickup borough
uber_df = uber_df.merge(borough[['LocationID', 'Borough', 'Zone']], 
                        left_on='PULocationID', right_on='LocationID', how='left')
uber_df.rename(columns={'Borough': 'PU_Borough', 
                        'Zone': 'PU_Zone', 
                        'LocationID': 'PU_LocationID'}, 
               inplace=True)
#merge dropoff locations
uber_df = uber_df.merge(borough[['LocationID', 'Borough', 'Zone']], 
                        left_on='DOLocationID', right_on='LocationID', how='left')
uber_df.rename(columns={'Borough': 'DO_Borough', 
                        'Zone': 'DO_Zone', 
                        'LocationID': 'DO_LocationID'}, 
               inplace=True)

In [5]:
uber_df = uber_df[(uber_df['trip_miles']>0.1)
                   & (uber_df['trip_time']>60)&
                      (uber_df['trip_miles']<100)].copy()
uber_df.head()

,hvfhs_license_num,request_datetime,trip_miles,trip_time,PULocationID,DOLocationID,congestion_surcharge,shared_request_flag,PU_LocationID,PU_Borough,PU_Zone,DO_LocationID,DO_Borough,DO_Zone
0,HV0003,2025-01-01 00:28:07,1.32,1259,148,211,2.75,N,148,Manhattan,Lower East Side,211,Manhattan,SoHo
1,HV0003,2025-01-01 00:28:22,13.43,2874,132,181,0.00,N,132,Queens,JFK Airport,181,Brooklyn,Park Slope
2,HV0003,2025-01-01 00:27:13,0.82,264,76,76,0.00,N,76,Brooklyn,East New York,76,Brooklyn,East New York
3,HV0003,2025-01-01 00:33:29,1.61,457,76,76,0.00,N,76,Brooklyn,East New York,76,Brooklyn,East New York
4,HV0003,2025-01-01 00:34:43,9.66,2073,112,89,0.00,N,112,Brooklyn,Greenpoint,89,Brooklyn,Flatbush/Ditmas Park


In [ ]:
#Time Bucket
uber_df['hour'] = uber_df['request_datetime'].dt.hour
uber_df['day_of_week'] = uber_df['request_datetime'].dt.dayofweek
uber_df['is_weekend'] = uber_df['day_of_week'].isin([4,5,6]).astype(int)

#bands
def time_band(hour):
    if 0 <= hour < 6:
        return 'Late Night'
    elif 6 <= hour < 10:
        return 'Morning'
    elif 10 <= hour < 16:
        return 'Midday'
    elif 16 <= hour < 20:
        return 'Evening'
    else:
        return 'Night'
uber_df['time_band'] = uber_df['hour'].apply(time_band)
print(uber_df[['hour', 'time_band']].head())

   hour   time_band
0     0  Late Night
1     0  Late Night
2     0  Late Night
3     0  Late Night
4     0  Late Night


In [8]:
uber_df['avg_speed_mph'] = uber_df['trip_miles'] / (uber_df['trip_time'] / 3600)
uber_df['trip_time_minutes'] = uber_df['trip_time'] / 60